# 📊 Notebook 1 — Data Collection from shorts_master_index.csv

This notebook uses your **shorts_master_index.csv** (1,145 real YouTube Shorts) to:
- Download videos in batches via yt-dlp
- Extract frames from each scene
- Compute CLIP embeddings
- Auto-label using **views signal + CLIP quality score**
- Use the `category` column directly as style labels

**What the CSV gives us for free:**
| Column | Used for |
|--------|----------|
| `url` | Download source |
| `category` | Direct style label (8 categories) |
| `views` | Quality signal — high views = better content |
| `duration` | Filter/sampling |

**Categories in your CSV:** anime, movie_villain, rap_hiphop, motivational, gaming, athlete, ufc_mma, aesthetic_flow

---
> ⚠️ **Start with 150 videos** (`MAX_VIDEOS_PER_RUN = 150`). Colab free tier can handle that in one session. Run again with more after training confirms results.

In [ ]:
# ── Install everything ──
!pip install -q open-clip-torch torch torchvision Pillow tqdm yt-dlp \
    opencv-python-headless scenedetect[opencv] pandas scikit-learn

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/video_ai_training'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/frames', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/videos', exist_ok=True)
print(f'Drive root ready: {DRIVE_ROOT}')

In [ ]:
# ── Configuration ──

# Path to your CSV on Google Drive (after you upload it)
CSV_PATH = '/content/drive/MyDrive/shorts_master_index.csv'

# How many videos to download per run
# Start with 150 — you can re-run with more later
MAX_VIDEOS_PER_RUN = 150

# Optional: filter to specific categories only (None = use all 8)
CATEGORIES_TO_USE = None  # e.g. ['anime', 'motivational', 'movie_villain']

# Scene / frame settings (tuned for Shorts — fast cuts)
SCENE_THRESHOLD   = 25.0   # lower = detects more cuts
MIN_SCENE_DURATION = 0.2   # seconds — Shorts cut very fast
FRAMES_PER_SCENE   = 2     # 2 frames per scene is enough
MAX_SCENES_PER_VIDEO = 60  # cap per video

# CLIP model
CLIP_MODEL      = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'

print('Config OK.')

In [ ]:
import pandas as pd

# Load the CSV
df = pd.read_csv(CSV_PATH)
print(f"Raw rows: {len(df)}")

# Drop duplicate videos (same video_id can appear in multiple categories)
df = df.drop_duplicates(subset='video_id', keep='first').reset_index(drop=True)
print(f"Unique videos: {len(df)}")

# Filter categories if requested
if CATEGORIES_TO_USE:
    df = df[df['category'].isin(CATEGORIES_TO_USE)].reset_index(drop=True)
    print(f"After category filter: {len(df)}")

# Overview
print("\nCategory breakdown:")
breakdown = df['category'].value_counts()
print(breakdown.to_string())

print("\nTop 5 by views:")
print(df.nlargest(5, 'views')[['title','category','views']].to_string(index=False))

In [ ]:
import math

categories = df['category'].unique().tolist()
per_cat = MAX_VIDEOS_PER_RUN // len(categories)
print(f"Categories: {len(categories)}  |  Videos per category: {per_cat}  |  Total target: {per_cat * len(categories)}")

# Take the top-N by views within each category for best quality signal
frames = []
for cat in categories:
    subset = df[df['category'] == cat].nlargest(per_cat, 'views')
    frames.append(subset)

df_download = pd.concat(frames).reset_index(drop=True)
print(f"\nFinal download list: {len(df_download)} videos")
print(df_download['category'].value_counts().to_string())

In [ ]:
import os, json, subprocess, time

VIDEO_DIR   = '/content/drive/MyDrive/training_data/videos'
LOG_PATH    = '/content/drive/MyDrive/training_data/download_log.json'

os.makedirs(VIDEO_DIR, exist_ok=True)

# Resume support — load existing log
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        download_log = json.load(f)
else:
    download_log = {}  # {video_id: local_path}

print(f"Already downloaded: {len(download_log)} videos")

downloaded_paths = {}  # video_id -> local_path (only new downloads this run)
failed           = []

for _, row in df_download.iterrows():
    vid = str(row['video_id'])
    url = str(row['url'])

    # Skip if already downloaded
    if vid in download_log and os.path.exists(download_log[vid]):
        downloaded_paths[vid] = download_log[vid]
        continue

    out_tmpl = os.path.join(VIDEO_DIR, f"{vid}.%(ext)s")
    cmd = [
        'yt-dlp',
        '--format', 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        '--merge-output-format', 'mp4',
        '--output', out_tmpl,
        '--no-playlist',
        '--socket-timeout', '30',
        '--retries', '2',
        '--quiet',
        url,
    ]
    try:
        result = subprocess.run(cmd, timeout=120, capture_output=True, text=True)
        out_path = os.path.join(VIDEO_DIR, f"{vid}.mp4")
        if os.path.exists(out_path):
            downloaded_paths[vid] = out_path
            download_log[vid]    = out_path
            print(f"  ✓ {vid}  [{row['category']}]")
        else:
            print(f"  ✗ {vid} — file not found after download")
            failed.append(vid)
    except subprocess.TimeoutExpired:
        print(f"  ✗ {vid} — timeout")
        failed.append(vid)
    except Exception as e:
        print(f"  ✗ {vid} — {e}")
        failed.append(vid)

# Save updated log
with open(LOG_PATH, 'w') as f:
    json.dump(download_log, f, indent=2)

total_available = len(download_log)
print(f"\nDownloaded this run : {len(downloaded_paths)}")
print(f"Failed this run     : {len(failed)}")
print(f"Total in log        : {total_available}")

In [ ]:
import cv2
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector

FRAMES_DIR   = '/content/drive/MyDrive/training_data/frames'
METADATA_PATH = '/content/drive/MyDrive/training_data/frame_metadata.json'
os.makedirs(FRAMES_DIR, exist_ok=True)

# Load existing metadata for resume support
if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH) as f:
        frame_metadata = json.load(f)
else:
    frame_metadata = []  # list of {frame_path, video_id, category, views, scene_idx, frame_idx}

processed_ids = {m['video_id'] for m in frame_metadata}

# Build lookup: video_id → {category, views}
meta_lookup = df_download.set_index('video_id')[['category','views']].to_dict('index')

new_frames = 0
for vid, path in download_log.items():
    if vid in processed_ids:
        continue
    if not os.path.exists(path):
        continue

    cat   = meta_lookup.get(vid, {}).get('category', 'unknown')
    views = int(meta_lookup.get(vid, {}).get('views', 0))

    try:
        video = open_video(path)
        sm    = SceneManager()
        sm.add_detector(ContentDetector(threshold=SCENE_THRESHOLD, min_scene_len=int(MIN_SCENE_DURATION * 30)))
        sm.detect_scenes(video)
        scenes = sm.get_scene_list()[:MAX_SCENES_PER_VIDEO]
    except Exception as e:
        print(f"  Scene detect failed {vid}: {e}")
        continue

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30

    for s_idx, (start, end) in enumerate(scenes):
        start_frame = start.get_frames()
        end_frame   = end.get_frames()
        sample_pts  = [
            start_frame + int((end_frame - start_frame) * t)
            for t in [0.25, 0.75]
        ][:FRAMES_PER_SCENE]

        for f_idx, fnum in enumerate(sample_pts):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)
            ret, frame = cap.read()
            if not ret:
                continue
            fname     = f"{vid}_s{s_idx:04d}_f{f_idx}.jpg"
            fpath     = os.path.join(FRAMES_DIR, fname)
            cv2.imwrite(fpath, frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
            frame_metadata.append({
                'frame_path': fpath,
                'video_id'  : vid,
                'category'  : cat,
                'views'     : views,
                'scene_idx' : s_idx,
                'frame_idx' : f_idx,
            })
            new_frames += 1

    cap.release()

# Save metadata
with open(METADATA_PATH, 'w') as f:
    json.dump(frame_metadata, f)

print(f"New frames extracted : {new_frames}")
print(f"Total frames in metadata : {len(frame_metadata)}")

In [ ]:
import torch
import open_clip
from PIL import Image

EMBED_PATH  = '/content/drive/MyDrive/training_data/embeddings.pt'
BATCH_SIZE  = 64
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAINED)
model = model.to(DEVICE).eval()

# Load existing embeddings for resume
if os.path.exists(EMBED_PATH):
    saved         = torch.load(EMBED_PATH, map_location='cpu')
    all_embeddings = saved['embeddings']   # shape (N, 512)
    all_frame_names = saved['frame_names'] # list[str]
else:
    all_embeddings  = torch.zeros(0, 512)
    all_frame_names = []

existing_set = set(all_frame_names)

# Only embed frames not yet processed
new_meta = [m for m in frame_metadata if m['frame_path'] not in existing_set]
print(f"New frames to embed: {len(new_meta)}")

new_embs = []
for i in range(0, len(new_meta), BATCH_SIZE):
    batch_meta = new_meta[i : i + BATCH_SIZE]
    imgs = []
    valid_meta = []
    for m in batch_meta:
        try:
            img = preprocess(Image.open(m['frame_path']).convert('RGB'))
            imgs.append(img)
            valid_meta.append(m)
        except Exception:
            pass

    if not imgs:
        continue

    batch_tensor = torch.stack(imgs).to(DEVICE)
    with torch.no_grad():
        emb = model.encode_image(batch_tensor).cpu().float()
        emb = emb / emb.norm(dim=-1, keepdim=True)  # L2 normalise
    new_embs.append(emb)
    for m in valid_meta:
        all_frame_names.append(m['frame_path'])

    if (i // BATCH_SIZE) % 10 == 0:
        print(f"  Embedded {i + len(imgs)}/{len(new_meta)} frames...")

if new_embs:
    new_tensor     = torch.cat(new_embs, dim=0)
    all_embeddings = torch.cat([all_embeddings, new_tensor], dim=0)

torch.save({'embeddings': all_embeddings, 'frame_names': all_frame_names}, EMBED_PATH)
print(f"Total embeddings saved: {all_embeddings.shape[0]}")

In [ ]:
import numpy as np

LABELS_PATH = '/content/drive/MyDrive/training_data/labels.csv'

# Build frame_path → metadata lookup
meta_by_path = {m['frame_path']: m for m in frame_metadata}

# CLIP-quality score: cosine similarity to a "high quality cinematic shot" prompt
quality_prompt = open_clip.tokenize(["a high quality cinematic shot, sharp, well-lit"]).to(DEVICE)
with torch.no_grad():
    text_emb = model.encode_text(quality_prompt).cpu().float()
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

all_embeddings_loaded = all_embeddings  # already loaded above
clip_scores = (all_embeddings_loaded @ text_emb.T).squeeze(-1).numpy()  # shape (N,)

# Build full label table
rows = []
for idx, frame_path in enumerate(all_frame_names):
    m = meta_by_path.get(frame_path, {})
    rows.append({
        'frame_path': frame_path,
        'video_id'  : m.get('video_id', ''),
        'category'  : m.get('category', 'unknown'),
        'views'     : m.get('views', 0),
        'clip_score': float(clip_scores[idx]),
    })

df_labels = pd.DataFrame(rows)

# Normalise each signal to [0, 1]
df_labels['clip_norm']  = (df_labels['clip_score'] - df_labels['clip_score'].min()) / \
                          (df_labels['clip_score'].max() - df_labels['clip_score'].min() + 1e-9)
df_labels['views_norm'] = np.log1p(df_labels['views'])
df_labels['views_norm'] = df_labels['views_norm'] / (df_labels['views_norm'].max() + 1e-9)

# Composite aesthetic score: 60% visual quality, 40% popularity signal
df_labels['aesthetic_score'] = 0.6 * df_labels['clip_norm'] + 0.4 * df_labels['views_norm']

# Category index (used as multi-class style label in Notebook 2)
CATEGORY_ORDER = ['anime','movie_villain','rap_hiphop','motivational','gaming','athlete','ufc_mma','aesthetic_flow']
cat_to_idx     = {c: i for i, c in enumerate(CATEGORY_ORDER)}
df_labels['category_idx'] = df_labels['category'].map(cat_to_idx).fillna(-1).astype(int)

df_labels.to_csv(LABELS_PATH, index=False)
print(f"Labels saved: {len(df_labels)} rows → {LABELS_PATH}")
print(df_labels[['category','aesthetic_score']].groupby('category').mean().sort_values('aesthetic_score', ascending=False))

## ✅ Data Collection Complete

All files are saved to your Google Drive under `MyDrive/training_data/`:

| File | Contents |
|------|----------|
| `videos/` | Downloaded MP4s (one per video_id) |
| `frames/` | Extracted JPEG frames |
| `download_log.json` | Resume log — already-downloaded video IDs |
| `frame_metadata.json` | Per-frame record: video_id, category, views, scene/frame index |
| `embeddings.pt` | CLIP ViT-B-32 embeddings for every frame |
| `labels.csv` | Frame-level labels: clip_score, views, aesthetic_score (0–1), category_idx |

### Category index used in training (Notebook 2)
```
0 = anime          4 = gaming
1 = movie_villain  5 = athlete
2 = rap_hiphop     6 = ufc_mma
3 = motivational   7 = aesthetic_flow
```

**Next step → open `02_train_aesthetic_ranker.ipynb`** to train the visual ranker + style classifier on these labels.